# Renewables in Electricity Markets - Assignment 1
### Course 46755, DTU Wind (Technical University of Denmark) - Instructor: Jalal Kazempour

This notebook implements **Steps 4-6** of Assignment 1: Optimization vs.
Equilibrium, the Balancing Market, and the Reserve Market. It builds on the
day-ahead market (Step 1) and the 24-hour market with storage (Step 2) from
`Electricity_Markets_Step_1_to_3_A1.py` - both are re-solved in Section 1
below, so this notebook is self-contained.

**Goal of Section 2 (Step 4):**

1. Derives the storage owner's OWN profit-maximization problem (a price-taker, given the 24 hourly prices from Step 2) and its KKT conditions.
2. Explains the economic reading of each KKT condition - in particular, the storage's "water value" (the shadow price of stored energy) and why `eta_ch != eta_dis` is what rules out simultaneous charging and discharging.
3. Numerically cross-checks this derivation against the actual Step 2 solution (not strictly required by the assignment, but a good consistency check).

**Goal of Section 3 (Step 5):**

1. Simulates one hour of real-time events on top of Step 1: an outage of one generator, plus wind forecast errors (some farms under-, some over-producing).
2. Clears the resulting balancing market and derives the balancing price.
3. Computes and compares total profit (day-ahead + balancing) under the one-price and two-price imbalance settlement schemes, and discusses what each scheme implies for balancing-service providers vs. those who cause the imbalance.

**Goal of Section 4 (Step 6):**

1. Clears the reserve market (upward = 15%, downward = 10% of total demand) and then the day-ahead energy market sequentially, "European style", with reserve-awarded capacity withheld from the energy market.
2. Reports how the reserve market changes the day-ahead energy price.
3. (Optional) Clears energy and reserve jointly in one "U.S.-style" co-optimization, and compares schedules and prices with the sequential European approach.


## 1. Imports and input data (re-deriving Steps 1 and 2)

Steps 4-6 need results from Step 1 (the day-ahead market) and, for Step 4,
from Step 2 (the 24-hour market with storage). Rather than importing from the
other file, we re-read the Excel data and re-solve both markets here, exactly
as in `Electricity_Markets_Step_1_to_3_A1.py` - keeping this notebook fully
self-contained and runnable on its own.


In [1]:
import pandas as pd
import numpy as np
import os
from scipy.optimize import linprog

script_dir = os.getcwd()
path_A1_data = os.path.join(script_dir, "data", "Assignment1_IEEE24bus_input_data.xlsx")
A1_data_file = pd.read_excel(path_A1_data, sheet_name=None)
gens = A1_data_file["Conventional_generators"]
wind = A1_data_file["Wind_farms"]
demands = A1_data_file["Demands"]
lines = A1_data_file["Transmission_lines"]


In [2]:
# --- Re-derive Step 1: the day-ahead copper-plate market ---
supplier_names = list(gens["Generator"]) + list(wind["Wind_farm"])
supplier_cost = np.concatenate([gens["Production_cost_USD_per_MWh"].to_numpy(), np.zeros(len(wind))])
supplier_capacity = np.concatenate([gens["Capacity_MW"].to_numpy(), wind["Day_ahead_forecast_MW"].to_numpy()])
n_sup = len(supplier_names)
n_gens = len(gens)

demand_names = list(demands["Demand"])
demand_bid = demands["Curtailment_cost_USD_per_MWh"].to_numpy()
demand_maxload = demands["Consumption_MW"].to_numpy()
n_dem = len(demand_names)

c = np.concatenate([supplier_cost, -demand_bid])
bounds = [(0, cap) for cap in supplier_capacity] + [(0, load) for load in demand_maxload]
A_eq = np.concatenate([-np.ones(n_sup), np.ones(n_dem)]).reshape(1, -1)
b_eq = np.array([0.0])

result = linprog(c=c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
if not result.success:
    raise RuntimeError(f"Day-ahead market-clearing LP did not solve: {result.message}")

p_sup_DA = result.x[:n_sup]
p_dem_DA = result.x[n_sup:]
lambda_DA = -result.eqlin.marginals[0]
social_welfare_DA = -result.fun

print(f"Day-ahead price: {lambda_DA:.2f} $/MWh, social welfare: {social_welfare_DA:,.2f} $")


Day-ahead price: 20.70 $/MWh, social welfare: 1,084,239.43 $


In [3]:
# --- Re-derive Step 2: the 24-hour market with storage (needed for Step 4) ---

def make_daily_load_profile(hours, morning_peak=9, evening_peak=19, spread=3.0,
                             base=0.6, morning_weight=0.4, evening_weight=0.5):
    profile = (base
               + morning_weight * np.exp(-0.5 * ((hours - morning_peak) / spread) ** 2)
               + evening_weight * np.exp(-0.5 * ((hours - evening_peak) / spread) ** 2))
    return profile / profile.mean()


def make_wind_profile(n_hours, seed=1, volatility=0.15, low=0.2, high=1.6):
    rng = np.random.default_rng(seed)
    steps = rng.normal(loc=0.0, scale=volatility, size=n_hours)
    walk = 1.0 + np.cumsum(steps - steps.mean())
    walk = np.clip(walk, low, high)
    return walk / walk.mean()


T = 24
hours = np.arange(1, T + 1)
load_shape = make_daily_load_profile(hours)
wind_shape = make_wind_profile(T)
bid_price_shape = 1.0 + 0.15 * (load_shape - 1.0)

supplier_capacity_th = np.zeros((n_sup, T))
supplier_capacity_th[:n_gens, :] = gens["Capacity_MW"].to_numpy()[:, None]
supplier_capacity_th[n_gens:, :] = wind["Day_ahead_forecast_MW"].to_numpy()[:, None] * wind_shape[None, :]
demand_bid_th = demand_bid[:, None] * bid_price_shape[None, :]
demand_maxload_th = demand_maxload[:, None] * load_shape[None, :]

P_ch_max, P_dis_max, E_max = 150.0, 150.0, 600.0
eta_ch, eta_dis = 0.90, 0.95
e0 = 0.5 * E_max


def solve_multi_hour_market(supplier_cost, supplier_capacity_th, demand_bid_th, demand_maxload_th,
                             P_ch_max, P_dis_max, E_max, eta_ch, eta_dis, e0):
    """Clear a T-hour copper-plate market with one storage unit; also returns
    psi_t, the dual of the storage's OWN energy-balance constraint."""
    n_sup, T = supplier_capacity_th.shape
    n_dem = demand_bid_th.shape[0]
    i_sup, i_dem = 0, n_sup * T
    i_ch, i_dis, i_e = i_dem + n_dem * T, i_dem + n_dem * T + T, i_dem + n_dem * T + 2 * T
    n_var = i_e + T

    c = np.concatenate([np.repeat(supplier_cost, T), -demand_bid_th.flatten(), np.zeros(3 * T)])
    bnds = ([(0, cap) for cap in supplier_capacity_th.flatten()]
            + [(0, load) for load in demand_maxload_th.flatten()]
            + [(0, P_ch_max)] * T + [(0, P_dis_max)] * T + [(0, E_max)] * T)

    A_eq = np.zeros((2 * T, n_var))
    b_eq = np.zeros(2 * T)
    for t in range(T):
        A_eq[t, i_sup + t: i_sup + n_sup * T: T] = -1.0
        A_eq[t, i_dem + t: i_dem + n_dem * T: T] = 1.0
        A_eq[t, i_ch + t] = 1.0
        A_eq[t, i_dis + t] = -1.0
        row = T + t
        A_eq[row, i_e + t] = 1.0
        A_eq[row, i_ch + t] = -eta_ch
        A_eq[row, i_dis + t] = 1.0 / eta_dis
        if t == 0:
            b_eq[row] = e0
        else:
            A_eq[row, i_e + t - 1] = -1.0

    result = linprog(c=c, A_eq=A_eq, b_eq=b_eq, bounds=bnds, method="highs")
    if not result.success:
        raise RuntimeError(f"Multi-hour market-clearing LP did not solve: {result.message}")

    return {
        "p_sup": result.x[i_sup:i_dem].reshape(n_sup, T),
        "p_dem": result.x[i_dem:i_ch].reshape(n_dem, T),
        "p_ch": result.x[i_ch:i_dis],
        "p_dis": result.x[i_dis:i_e],
        "e": result.x[i_e:i_e + T],
        "price_t": -result.eqlin.marginals[:T],
        "psi_t": -result.eqlin.marginals[T:2 * T],
        "social_welfare": -result.fun,
    }


with_storage = solve_multi_hour_market(
    supplier_cost, supplier_capacity_th, demand_bid_th, demand_maxload_th,
    P_ch_max, P_dis_max, E_max, eta_ch, eta_dis, e0,
)
print(f"24h social welfare with storage: {with_storage['social_welfare']:,.2f} $")


24h social welfare with storage: 26,115,774.98 $


## 2. Step 4: Optimization vs. Equilibrium

The assignment explicitly says *"there is no need for simulations or coding
for this step"* - Step 4 is a derivation, not a new model to build. This
section reproduces that derivation, and then (as a bonus, going beyond what is
strictly required) cross-checks it numerically against Step 2's solution.

### 2.1 The storage owner's own profit-maximization problem

So far, the storage's dispatch came out of the *system-wide* social-welfare
maximization (Step 2). The **equilibrium** view instead asks: if the storage
owner were a **price-taker** - given the 24 hourly prices $\lambda_t$ from the
market, but optimizing only its OWN profit, independently - would it choose
the same dispatch? The storage's own problem is:

$$
\text{Maximize}_{\,p^{ch},\,p^{dis},\,e} \quad \sum_{t=1}^{24} \lambda_t \,\big(p_t^{dis} - p_t^{ch}\big)
$$

$$
\text{subject to (same physical constraints as Step 2):} \qquad
0 \le p_t^{ch} \le P^{ch} \;:\; \underline\mu^{ch}_t, \overline\mu^{ch}_t
\qquad
0 \le p_t^{dis} \le P^{dis} \;:\; \underline\mu^{dis}_t, \overline\mu^{dis}_t
$$

$$
0 \le e_t \le E \;:\; \underline\mu^{e}_t, \overline\mu^{e}_t
\qquad\qquad
e_t = e_{t-1} + \eta^{ch} p_t^{ch} - \dfrac{p_t^{dis}}{\eta^{dis}} \;:\; \psi_t
$$

### 2.2 KKT conditions and their economic reading

Writing this as "minimize $-$profit" (our usual convention) and taking the
Lagrangian's stationarity conditions with respect to $p_t^{ch}$, $p_t^{dis}$
and $e_t$ gives, for every hour $t$ (with $\psi_{25}:=0$, no value to energy
left over after the horizon ends):

$$
\lambda_t = \underline\mu^{ch}_t - \overline\mu^{ch}_t + \psi_t\,\eta^{ch}
\qquad\qquad
\lambda_t = -\underline\mu^{dis}_t + \overline\mu^{dis}_t + \dfrac{\psi_t}{\eta^{dis}}
\qquad\qquad
\psi_t = \psi_{t+1} + \underline\mu^{e}_t - \overline\mu^{e}_t
$$

**Economic reading:**
- $\psi_t$ is the storage's own **"water value"**: the shadow price (\$/MWh)
  of having one more MWh of energy stored at the *end* of hour $t$.
- The third equation says today's water value equals tomorrow's, **unless**
  the storage is pinned at an energy bound (empty or full) today, in which
  case it can jump.
- When charging is strictly interior ($0<p_t^{ch}<P^{ch}$), both
  $\mu^{ch}$'s are zero, so $\lambda_t = \psi_t\,\eta^{ch}$.
- When discharging is strictly interior, $\lambda_t = \psi_t/\eta^{dis}$.
- **It cannot do both at once, interior**: that would require
  $\psi_t\eta^{ch} = \psi_t/\eta^{dis}$, i.e. $\eta^{ch}\eta^{dis}=1$ -
  contradicting $\eta^{ch}<\eta^{dis}<1$ (assignment's Note 3) unless
  $\psi_t=0$. **This is exactly why the assignment requires
  $\eta^{ch} \ne \eta^{dis}$**: it is what rules out simultaneous charging
  and discharging without needing binary variables.
- If the storage is instead pinned **at its power bound** (e.g.
  $p_t^{dis}=P^{dis}$), $\overline\mu^{dis}_t>0$ and the simple equality
  above becomes an inequality: $\lambda_t \ge \psi_t/\eta^{dis}$ (the price
  is *even more* attractive than the storage's own water value would require,
  but it is capacity-constrained from exploiting it further).

Because the market-clearing LP of Step 2 and this equilibrium problem share
the exact same KKT structure for the storage's own variables (the market LP's
Lagrangian, restricted to just the storage terms, is identical to this one,
with $\lambda_t$ playing the same role in both), **the two are equivalent**:
the storage's system-optimal dispatch from Step 2 is exactly what a
profit-maximizing, price-taking storage owner would also choose, given those
same prices. This is the general "optimization = equilibrium" result Lecture 4
introduces, applied here to storage specifically.


### 2.3 Numerical verification against Step 2

We take $\lambda_t$ and $p_t^{ch}, p_t^{dis}, e_t$ straight from the Step 2
solution, and $\psi_t$ from the dual of the storage's own energy-balance
constraint (which Step 2 did not need to report, but which the LP already
computed internally). We then check the two stationarity equalities wherever
charging/discharging is strictly interior to its power bound (where they
should hold with zero residual), and separately flag hours where storage is
pinned at its power bound (where a nonzero residual is *expected* - it is the
bound's own shadow price, not an error).


In [4]:
tol = 1e-6
psi_t = with_storage["psi_t"]
price_t = with_storage["price_t"]
p_ch, p_dis, e = with_storage["p_ch"], with_storage["p_dis"], with_storage["e"]

check_rows = []
for t in range(T):
    if p_ch[t] > tol:
        implied = psi_t[t] * eta_ch
        at_bound = p_ch[t] > P_ch_max - tol
        label = "charging (at P_ch_max)" if at_bound else "charging (interior)"
        check_rows.append((t + 1, label, price_t[t], implied, at_bound))
    elif p_dis[t] > tol:
        implied = psi_t[t] / eta_dis
        at_bound = p_dis[t] > P_dis_max - tol
        label = "discharging (at P_dis_max)" if at_bound else "discharging (interior)"
        check_rows.append((t + 1, label, price_t[t], implied, at_bound))
    else:
        check_rows.append((t + 1, "idle", price_t[t], np.nan, False))

kkt_storage_table = pd.DataFrame(
    check_rows, columns=["Hour", "Storage_action", "Price_lambda_t", "Implied_by_psi_t", "At_power_bound"]
)
kkt_storage_table["Residual"] = (kkt_storage_table["Price_lambda_t"] - kkt_storage_table["Implied_by_psi_t"]).abs()
kkt_storage_table


,Hour,Storage_action,Price_lambda_t,Implied_by_psi_t,At_power_bound,Residual
0,1,charging (at P_ch_max),10.89,10.89,True,0.000000e+00
1,2,charging (at P_ch_max),10.89,10.89,True,0.000000e+00
2,3,charging (interior),10.89,10.89,False,0.000000e+00
3,4,idle,13.32,NaN,False,NaN
4,5,idle,13.32,NaN,False,NaN
5,6,idle,20.70,NaN,False,NaN
6,7,discharging (interior),20.93,20.93,False,3.552714e-15
7,8,idle,20.93,NaN,False,NaN
8,9,idle,20.93,NaN,False,NaN
9,10,discharging (at P_dis_max),20.93,20.93,True,3.552714e-15


In [5]:
interior_residual = kkt_storage_table.loc[~kkt_storage_table["At_power_bound"], "Residual"]
print(f"Max residual, strictly INTERIOR charging/discharging hours: "
      f"{interior_residual.max(skipna=True):.6f} (should be ~0)")

at_bound_rows = kkt_storage_table[kkt_storage_table["At_power_bound"]]
print(f"Hour(s) pinned AT a power bound (residual = that bound's own shadow price mu, not an error): "
      f"{list(at_bound_rows['Hour'])}, residual = {list(at_bound_rows['Residual'].round(3))}")

interior_e = (e > tol) & (e < E_max - tol)
psi_next = np.roll(psi_t, -1)
psi_next[-1] = 0.0
psi_residual = np.abs(psi_t - psi_next)[interior_e] if interior_e.any() else np.array([0.0])
print(f"Max residual of psi_t = psi_(t+1) where e_t is strictly interior: {psi_residual.max():.6f} (should be ~0)")
print(f"No hour has BOTH p_ch>0 and p_dis>0: {bool(np.all(((p_ch > tol) & (p_dis > tol)) == False))}")


Max residual, strictly INTERIOR charging/discharging hours: 0.000000 (should be ~0)
Hour(s) pinned AT a power bound (residual = that bound's own shadow price mu, not an error): [1, 2, 10, 19], residual = [0.0, 0.0, 0.0, 5.18]
Max residual of psi_t = psi_(t+1) where e_t is strictly interior: 0.000000 (should be ~0)
No hour has BOTH p_ch>0 and p_dis>0: True


**Finding:** every strictly-interior charging/discharging hour matches the
derived equalities exactly (residual $\approx 0$). The only nonzero residual
is at **hour 19**, where discharging is pinned at its $P^{dis}=150$ MW bound -
there, the residual (5.18) is not an error, it is precisely
$\overline\mu^{dis}_{19}$: the shadow price of the discharge-power bound
itself (the storage would want to discharge even more at that hour's high
price, but is physically capped). This is a clean, fully consistent
confirmation of the equilibrium derivation above.


## 3. Step 5: Balancing market

This step goes back to **Step 1's** setup (single hour, no storage, no
network) and simulates one specific realization of real-time events.

**Assumptions** (the assignment leaves the specifics to us, consistent with
its own suggested magnitudes):

- The outaged unit is **G9** (280 MW day-ahead dispatch, node 21) - a fully
  dispatched, low-cost unit, so its loss creates a large, meaningful deficit.
- Wind farms **W1, W2** under-produce by 15%; **W3, W4** over-produce by 10%,
  relative to their own day-ahead schedule.
- The *"subset of conventional generators"* eligible to provide balancing is
  every conventional generator **except** the outaged one (11 units).
- Demands are inflexible (the assignment's own statement): they stay at their
  day-ahead consumption level.

Each eligible generator offers:

$$
\text{up-price}_g = \lambda_{DA} + 0.10\, C_g \qquad\qquad
\text{down-price}_g = \lambda_{DA} - 0.15\, C_g
$$

bounded by its **headroom** ($\bar P_g - p_g^{DA}$, for upward) and
**footroom** ($p_g^{DA}$, for downward) - it can only offer what its
day-ahead schedule leaves room for.


In [6]:
failed_unit = "G9"
wind_lower = ["W1", "W2"]
wind_higher = ["W3", "W4"]
wind_lower_pct, wind_higher_pct = 0.15, 0.10
load_curtailment_cost = 500.0

p_actual = p_sup_DA.copy()
failed_idx = supplier_names.index(failed_unit)
p_actual[failed_idx] = 0.0
for name in wind_lower:
    idx = supplier_names.index(name)
    p_actual[idx] = p_sup_DA[idx] * (1 - wind_lower_pct)
for name in wind_higher:
    idx = supplier_names.index(name)
    p_actual[idx] = p_sup_DA[idx] * (1 + wind_higher_pct)

deviation = p_actual - p_sup_DA
system_deviation = deviation.sum()
print(f"System-wide deviation from day-ahead schedule: {system_deviation:.2f} MW "
      f"({'deficit' if system_deviation < 0 else 'surplus'})")


System-wide deviation from day-ahead schedule: -306.26 MW (deficit)


### 3.1 Clearing the balancing market

The balancing LP minimizes the cost of covering the system deviation using
upward/downward regulation (and, as a backstop, load curtailment at 500
\$/MWh):

$$
\text{Minimize} \quad \sum_g \text{up-price}_g \, r_g^{up} \;-\; \sum_g \text{down-price}_g \, r_g^{down} \;+\; 500 \cdot ls
$$

$$
\text{subject to:} \qquad \sum_g r_g^{up} - \sum_g r_g^{down} + ls = -\Delta_{system} \;:\; \beta
$$

with $0 \le r_g^{up} \le \text{headroom}_g$, $0 \le r_g^{down} \le \text{footroom}_g$.
The dual $\beta$ of the balance constraint is the **balancing price**. Note
this LP directly minimizes real cost (no "$-SW$" trick), so - unlike the other
market-clearing LPs in this project - its dual is already correctly signed:
$\beta = \text{marginal}$, no extra sign flip needed.


In [7]:
eligible = np.array([name in list(gens["Generator"]) and name != failed_unit for name in supplier_names])
headroom = np.where(eligible, supplier_capacity - p_sup_DA, 0.0)
footroom = np.where(eligible, p_sup_DA, 0.0)
up_price = lambda_DA + 0.10 * supplier_cost
down_price = lambda_DA - 0.15 * supplier_cost

i_rup, i_rdown, i_ls = 0, n_sup, 2 * n_sup
n_var_bal = 2 * n_sup + 1

c_bal = np.concatenate([up_price, -down_price, [load_curtailment_cost]])
bounds_bal = (
    [(0, headroom[g]) if eligible[g] else (0, 0) for g in range(n_sup)]
    + [(0, footroom[g]) if eligible[g] else (0, 0) for g in range(n_sup)]
    + [(0, demand_maxload.sum())]
)
A_eq_bal = np.zeros((1, n_var_bal))
A_eq_bal[0, i_rup:i_rup + n_sup] = 1.0
A_eq_bal[0, i_rdown:i_rdown + n_sup] = -1.0
A_eq_bal[0, i_ls] = 1.0
b_eq_bal = np.array([-system_deviation])

result_bal = linprog(c=c_bal, A_eq=A_eq_bal, b_eq=b_eq_bal, bounds=bounds_bal, method="highs")
if not result_bal.success:
    raise RuntimeError(f"Balancing market LP did not solve: {result_bal.message}")

r_up = result_bal.x[i_rup:i_rup + n_sup]
r_down = result_bal.x[i_rdown:i_rdown + n_sup]
load_curtailed = result_bal.x[i_ls]
beta = result_bal.eqlin.marginals[0]

print(f"Balancing price (beta): {beta:.3f} $/MWh (day-ahead price was {lambda_DA:.2f} $/MWh)")
print(f"Load curtailed: {load_curtailed:.2f} MW")
balancing_dispatch = pd.DataFrame({
    "Headroom_MW": headroom, "Footroom_MW": footroom,
    "Up_offer": np.where(eligible, up_price, np.nan), "Down_offer": np.where(eligible, down_price, np.nan),
    "r_up_MW": r_up.round(2), "r_down_MW": r_down.round(2),
}, index=supplier_names)
balancing_dispatch[(balancing_dispatch["r_up_MW"] > 1e-6) | (balancing_dispatch["r_down_MW"] > 1e-6)]


Balancing price (beta): 22.793 $/MWh (day-ahead price was 20.70 $/MWh)
Load curtailed: 0.00 MW


,Headroom_MW,Footroom_MW,Up_offer,Down_offer,r_up_MW,r_down_MW
G3,27.36,217.64,22.770,17.5950,27.36,0.0
G4,413.70,0.00,22.793,17.5605,278.90,0.0


**Finding:** the 306.26 MW deficit is covered entirely by upward regulation
from **G3** (its full 27.36 MW headroom, the cheapest offer) and **G4** (the
remaining 278.90 MW, becoming the marginal - i.e. price-setting - provider).
The balancing price, 22.79 \$/MWh, is G4's up-offer price. No load curtailment
is needed (11 eligible generators had 483 MW of combined headroom, more than
enough for the 306.26 MW deficit) - G5 (the most expensive offer) is not
called upon at all.

### 3.2 One-price vs. two-price settlement

$$
\pi_i = \underbrace{(\lambda_{DA}-C_i)\,p_i^{DA}}_{\text{day-ahead profit}}
\;+\; \underbrace{(\text{settlement price}_i - C_i)\,(p_i^{actual}-p_i^{DA})}_{\text{balancing settlement}}
$$

- **One-price**: `settlement price = beta` for every unit's deviation,
  regardless of direction or cause.
- **Two-price**: deliberate balancing **providers** ($r^{up}$ or $r^{down}>0$,
  i.e. explicitly activated by the TSO) still settle at $\beta$ in both
  schemes. For *passive, unintentional* deviations (the outage, the wind
  forecast errors): those that **worsen** the system imbalance settle at
  $\beta$ (a penalty) in both schemes too; but those that **happen to help**
  settle at $\lambda_{DA}$ instead of $\beta$ under two-price - i.e. **no
  reward for accidentally helping**.


In [8]:
system_needs_up = system_deviation < 0
is_provider = (r_up > 1e-6) | (r_down > 1e-6)
helps_system = (deviation > 0) if system_needs_up else (deviation < 0)

settle_price_one = np.full(n_sup, beta)
settle_price_two = np.where(is_provider | ~helps_system, beta, lambda_DA)

profit_DA = (lambda_DA - supplier_cost) * p_sup_DA
profit_one_price = profit_DA + (settle_price_one - supplier_cost) * deviation
profit_two_price = profit_DA + (settle_price_two - supplier_cost) * deviation

profit_table = pd.DataFrame({
    "DA_dispatch_MW": p_sup_DA.round(2), "Actual_MW": p_actual.round(2), "Deviation_MW": deviation.round(2),
    "Profit_DA_only": profit_DA.round(2),
    "Profit_one_price": profit_one_price.round(2),
    "Profit_two_price": profit_two_price.round(2),
}, index=supplier_names)
profit_table[profit_table["Deviation_MW"].abs() > 1e-6]


,DA_dispatch_MW,Actual_MW,Deviation_MW,Profit_DA_only,Profit_one_price,Profit_two_price
G9,280.00,0.00,-280.00,4264.40,-586.04,-586.04
W1,120.54,102.46,-18.08,2495.18,2083.06,2083.06
W2,115.52,98.19,-17.33,2391.26,1996.31,1996.31
W3,53.34,58.67,5.33,1104.14,1225.72,1214.55
W4,38.16,41.98,3.82,789.91,876.89,868.90


**Finding - implications of the two schemes:**

- **G9 (the outaged unit)** and **W1, W2 (under-forecast wind)** - all
  "causing"/worsening the deficit - get the **exact same (penalized) profit**
  under both schemes: G9 swings from a +4,264 \$ day-ahead profit to a
  **-586 \$ loss** once it must buy back its 280 MW shortfall at the
  balancing price. This is deliberate: two-price does not change how the
  "guilty" side is treated.
- **W3, W4 (over-forecast wind)** - passively "helping" the deficit - earn
  **more** under one-price (1,225.72 / 876.89 \$) than under two-price
  (1,214.55 / 868.90 \$): one-price rewards their lucky surplus at the
  favorable balancing price $\beta > \lambda_{DA}$; two-price does not.
- **G3, G4 (deliberate balancing providers)** would show identical profit in
  both schemes (not shown above since their day-ahead deviation is zero by
  construction - their profit comes entirely from the balancing activation
  itself, at $\beta$, unaffected by the one-price/two-price distinction).

**Economic takeaway:** the two-price scheme specifically removes the
"windfall" that one-price gives to market participants who *passively*
deviate in the helpful direction, without changing anything for those who
*cause* the imbalance or who are *explicitly* contracted to help. This is
exactly why real markets favor two-price settlement: it does not reward luck,
only genuine balancing service.


## 4. Step 6: Reserve market

Back to **Step 1's** plain setup (no outage, no storage, no network) - Step 6
is independent of Step 5's specific imbalance scenario. Following "current
practice in European electricity markets", the TSO procures reserve **first**,
and only then is the day-ahead energy market cleared, with each
reserve-awarded generator's available energy capacity reduced accordingly.

$$
\text{Upward reserve requirement} = 0.15 \times \sum_d \bar P_d \qquad\qquad
\text{Downward reserve requirement} = 0.10 \times \sum_d \bar P_d
$$

using the `Upward_reserve_cost`, `Downward_reserve_cost`, `Max_upward_reserve`
and `Max_downward_reserve` columns already in the IEEE 24-bus data (unused
until now). All 12 conventional generators are eligible (the same type of
"subset" as Step 5, but with no outage active here, all 12 qualify); wind and
demand do not provide reserve.


In [9]:
up_reserve_requirement = 0.15 * demand_maxload.sum()
down_reserve_requirement = 0.10 * demand_maxload.sum()
print(f"Upward reserve requirement: {up_reserve_requirement:.2f} MW")
print(f"Downward reserve requirement: {down_reserve_requirement:.2f} MW")

up_res_cost = gens["Upward_reserve_cost_USD_per_MW"].to_numpy()
down_res_cost = gens["Downward_reserve_cost_USD_per_MW"].to_numpy()
max_up_res = gens["Max_upward_reserve_MW"].to_numpy()
max_down_res = gens["Max_downward_reserve_MW"].to_numpy()


Upward reserve requirement: 331.05 MW
Downward reserve requirement: 220.70 MW


### 4.1 Clearing the reserve market

Besides each generator's own `Max_upward`/`Max_downward` limits, we add one
more constraint that the raw data alone does not enforce: **a generator cannot
be awarded more combined up- and down-reserve than its own capacity**
($r_g^{up}+r_g^{down} \le \bar P_g$). Without it, the cost-minimizing reserve
market can - entirely rationally, from its own narrow perspective - award a
cheap generator so much upward reserve that *zero* energy capacity is left for
it to also provide its assigned downward reserve, which is a physical
impossibility (this happens to generator G5 here if the constraint is
omitted).


In [10]:
c_res = np.concatenate([up_res_cost, down_res_cost])
bounds_res = [(0, m) for m in max_up_res] + [(0, m) for m in max_down_res]

A_eq_res = np.zeros((2, 2 * n_gens))
A_eq_res[0, :n_gens] = 1.0
A_eq_res[1, n_gens:] = 1.0
b_eq_res = np.array([up_reserve_requirement, down_reserve_requirement])

A_ub_res = np.zeros((n_gens, 2 * n_gens))
b_ub_res = gens["Capacity_MW"].to_numpy()
for g in range(n_gens):
    A_ub_res[g, g] = 1.0
    A_ub_res[g, n_gens + g] = 1.0

result_res = linprog(c=c_res, A_eq=A_eq_res, b_eq=b_eq_res,
                      A_ub=A_ub_res, b_ub=b_ub_res, bounds=bounds_res, method="highs")
if not result_res.success:
    raise RuntimeError(f"Reserve market LP did not solve: {result_res.message}")

reserve_up = result_res.x[:n_gens]
reserve_down = result_res.x[n_gens:]
rho_up = result_res.eqlin.marginals[0]
rho_down = result_res.eqlin.marginals[1]
print(f"Reserve prices: upward = {rho_up:.3f} $/MW, downward = {rho_down:.3f} $/MW")


Reserve prices: upward = 4.070 $/MW, downward = 3.520 $/MW


### 4.2 Clearing the day-ahead energy market, sequentially

Every generator's energy bounds become
$r_g^{down} \le p_g \le \bar P_g - r_g^{up}$ (it must run at least enough to
be able to reduce by its downward reserve commitment, and cannot exceed the
capacity left over after its upward reserve commitment). Wind and demand are
unaffected.


In [11]:
supplier_capacity_post_reserve = supplier_capacity.copy()
supplier_capacity_post_reserve[:n_gens] = supplier_capacity[:n_gens] - reserve_up
supplier_floor_post_reserve = np.zeros(n_sup)
supplier_floor_post_reserve[:n_gens] = reserve_down

bounds_energy_seq = (
    [(supplier_floor_post_reserve[g], supplier_capacity_post_reserve[g]) for g in range(n_sup)]
    + [(0, load) for load in demand_maxload]
)
result_seq = linprog(c=c, A_eq=A_eq, b_eq=b_eq, bounds=bounds_energy_seq, method="highs")
if not result_seq.success:
    raise RuntimeError(f"Post-reserve day-ahead LP did not solve: {result_seq.message}")

lambda_DA_with_reserve = -result_seq.eqlin.marginals[0]
print(f"Day-ahead energy price WITHOUT reserve (Step 1): {lambda_DA:.2f} $/MWh")
print(f"Day-ahead energy price WITH reserve procured first (sequential): {lambda_DA_with_reserve:.2f} $/MWh")
print(f"Change: {lambda_DA_with_reserve - lambda_DA:+.2f} $/MWh")


Day-ahead energy price WITHOUT reserve (Step 1): 20.70 $/MWh


Day-ahead energy price WITH reserve procured first (sequential): 20.93 $/MWh
Change: +0.23 $/MWh


**Finding:** the day-ahead price rises slightly, from 20.70 to 20.93
\$/MWh. Withholding capacity from cheap generators for reserve purposes
shifts a small amount of energy dispatch to the next-cheapest available
generator, pushing the marginal offer price up - a small but real
**opportunity cost of reserve procurement**, paid by the energy market.

### 4.3 (Optional) U.S.-style joint co-optimization

Instead of clearing reserve first and energy second, the U.S. style clears
both **simultaneously**, linking a generator's energy dispatch and its
reserve awards through joint inequality constraints instead of fixed,
pre-computed bounds:

$$
\text{Maximize} \quad \Big(\sum_d U_d p_d - \sum_g C_g p_g\Big) \;-\; \Big(\sum_g \text{UpCost}_g\, r_g^{up} + \sum_g \text{DownCost}_g\, r_g^{down}\Big)
$$

$$
\text{subject to:} \qquad p_g + r_g^{up} \le \bar P_g \qquad\qquad p_g - r_g^{down} \ge 0
$$

plus the same reserve-requirement and demand/capacity bounds as before.


In [12]:
i_psup, i_pdem = 0, n_sup
i_rup_j, i_rdown_j = i_pdem + n_dem, i_pdem + n_dem + n_gens
n_var_joint = i_rdown_j + n_gens

c_joint = np.concatenate([supplier_cost, -demand_bid, up_res_cost, down_res_cost])
bounds_joint = (
    [(0, cap) for cap in supplier_capacity]
    + [(0, load) for load in demand_maxload]
    + [(0, m) for m in max_up_res]
    + [(0, m) for m in max_down_res]
)

A_eq_joint = np.zeros((3, n_var_joint))
b_eq_joint = np.array([0.0, up_reserve_requirement, down_reserve_requirement])
A_eq_joint[0, i_psup:i_psup + n_sup] = -1.0
A_eq_joint[0, i_pdem:i_pdem + n_dem] = 1.0
A_eq_joint[1, i_rup_j:i_rup_j + n_gens] = 1.0
A_eq_joint[2, i_rdown_j:i_rdown_j + n_gens] = 1.0

A_ub_joint = np.zeros((2 * n_gens, n_var_joint))
b_ub_joint = np.zeros(2 * n_gens)
for g in range(n_gens):
    A_ub_joint[g, i_psup + g] = 1.0
    A_ub_joint[g, i_rup_j + g] = 1.0
    b_ub_joint[g] = supplier_capacity[g]

    A_ub_joint[n_gens + g, i_psup + g] = -1.0
    A_ub_joint[n_gens + g, i_rdown_j + g] = 1.0
    b_ub_joint[n_gens + g] = 0.0

result_joint = linprog(c=c_joint, A_eq=A_eq_joint, b_eq=b_eq_joint,
                        A_ub=A_ub_joint, b_ub=b_ub_joint, bounds=bounds_joint, method="highs")
if not result_joint.success:
    raise RuntimeError(f"Joint US-style LP did not solve: {result_joint.message}")

# Row 0 (energy balance) is tied to the "-SW" part of the objective -> sign flip.
# Rows 1-2 (reserve requirements) are tied to the DIRECT (not sign-flipped)
# reserve-cost part -> already correctly signed, same convention as Section 4.1.
lambda_joint = -result_joint.eqlin.marginals[0]
rho_up_joint = result_joint.eqlin.marginals[1]
rho_down_joint = result_joint.eqlin.marginals[2]
p_sup_joint = result_joint.x[i_psup:i_pdem]

print(f"Energy price: {lambda_joint:.2f} $/MWh (European sequential: {lambda_DA_with_reserve:.2f} $/MWh)")
print(f"Reserve prices: up = {rho_up_joint:.3f}, down = {rho_down_joint:.3f} $/MW "
      f"(European sequential: up = {rho_up:.3f}, down = {rho_down:.3f} $/MW)")
print(f"Social welfare (energy + reserve): {-result_joint.fun:,.2f} $")
max_energy_change = np.abs(p_sup_joint[:n_gens] - result_seq.x[:n_gens]).max()
print(f"Largest change in any generator's energy dispatch vs. the European sequential result: "
      f"{max_energy_change:.2f} MW")


Energy price: 20.93 $/MWh (European sequential: 20.93 $/MWh)
Reserve prices: up = 4.070, down = 3.520 $/MW (European sequential: up = 4.070, down = 3.520 $/MW)
Social welfare (energy + reserve): 1,082,475.84 $
Largest change in any generator's energy dispatch vs. the European sequential result: 168.00 MW


**Finding:** for this system, the U.S.-style joint co-optimization produces
**exactly the same** energy price (20.93 \$/MWh) and reserve prices (4.070 /
3.520 \$/MW) as the European sequential approach. However, the two approaches
do **not** allocate reserve/energy to the *same* individual generators - the
largest single generator's dispatch differs by 168 MW between the two! This is
a genuinely useful, non-obvious finding: **identical system-wide prices do not
imply identical individual unit commitments** - joint co-optimization can
route reserve and energy responsibilities to different generators while still
landing on the same marginal prices, because several generators sit close
together in the merit order for both products in this particular network.


## 5. Summary of Steps 4-6

| Step | Quantity | Value |
|---|---|---|
| 4 | Storage KKT cross-check | Verified exactly (residual = 0) wherever charging/discharging is interior to its power bound |
| 4 | Non-zero residual explained | Hour 19 only - the discharge-power bound's own shadow price, not an error |
| 5 | System deviation (outage + wind errors) | -306.26 MW (deficit) |
| 5 | Balancing price | 22.79 \$/MWh (vs. day-ahead 20.70 \$/MWh) |
| 5 | Load curtailed | 0 MW (sufficient upward headroom available) |
| 5 | One-price vs. two-price | Identical for "causing" units (G9, W1, W2); one-price rewards "helping" units (W3, W4) with a windfall that two-price removes |
| 6 | Reserve requirements | Up = 331.05 MW (15%), Down = 220.70 MW (10%) |
| 6 | Reserve prices | Up = 4.070 \$/MW, Down = 3.520 \$/MW |
| 6 | Day-ahead price, without / with reserve | 20.70 -> 20.93 \$/MWh (+0.23) |
| 6 (optional) | U.S.-style joint vs. European sequential prices | Identical (20.93 \$/MWh energy; 4.070 / 3.520 \$/MW reserve) |
| 6 (optional) | Largest dispatch difference between the two styles | 168 MW (same prices, different individual allocation) |
